In [41]:
!pip install qiskit
!pip install qiskit-aer
!pip install qiskit-ibm-runtime

In [42]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EnvironmentOptions, EstimatorOptions, SamplerOptions
from qiskit_aer import AerSimulator
import numpy as np
from numpy import pi
from matplotlib import pyplot as plt
import matplotlib
from scipy.optimize import minimize
from qiskit.circuit.library import QAOAAnsatz, hamiltonian_variational_ansatz, XXPlusYYGate, CPhaseGate, UnitaryGate
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.circuit import Parameter
from scipy.optimize import minimize
from scipy.linalg import eigh
from scipy.special import erf
from functools import partial
from qiskit_aer import AerSimulator
from qiskit.circuit.classical import expr
import random
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp
from qiskit.synthesis.evolution import SuzukiTrotter
from qiskit.quantum_info import random_statevector

## Measurement


In [43]:
def measure_ZZ(qc, q0, q1, cbit):
    qc.cx(q0, q1)
    qc.measure(q1, cbit)
    qc.cx(q0, q1)

In [44]:
def measure_XI(qc, q0, q1, cbit):
    qc.h(q0)
    qc.measure(q0, cbit)
    qc.h(q0)

**Important: fix convention $ Y = S X S^\dagger$. Later formulas must use the same convention to ensure a correct pauli tracking**

In [45]:
def measure_YI(qc, q0, q1, cbit):
    qc.sdg(q0)
    measure_XI(qc, q0, q1, cbit)
    qc.s(q0)

In [46]:
def measure_ZY(qc, q0, q1, cbit):

    # with the same convention, Y = S X S^d = S H Z H S^d
    qc.sdg(q1)
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)
    qc.s(q1)

In [47]:
def measure_ZX(qc, q0, q1, cbit):

    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)

## Single qubit clifford group

In [48]:
def add_H(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])
    measure_ZY(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.y(d)

    qc.x(d)

    qc.reset(a) # easy to check with gate-based circuit

    return qc

In [49]:
def add_S(qc, d, a, cbit):
    c = cbit

    # (ancilla, data) = (q+1, q)

    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    # measurement bit c_i encodes s_i = (-1)^{c_i}
    # s_i s_j = (-1)^{c[i] + c[j]}
    # product becomes XOR

    # Z^{(1 + s0 s1 s2)/2}
    # exponent = 1 when s0 s1 s2 = +1
    # s0 s1 s2 = (-1)^{c0 + c1 + c2}
    # +1 when (c0 + c1 + c2) mod 2 = 0 (even parity)

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.z(d)

    qc.reset(a) # easy to check with gate-based circuit

    return qc

In [50]:
def add_SH(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])  # s0
    measure_ZZ(qc, a, d, c[1])  # s1
    measure_ZY(qc, a, d, c[2])  # s2
    measure_YI(qc, a, d, c[3])  # s3
    measure_XI(qc, a, d, c[4])  # s4

    # X^{(1 + s0 s2 s3)/2} even parity
    parity_023 = expr.bit_xor(expr.bit_xor(c[0], c[2]), c[3])
    with qc.if_test(parity_023):
        qc.y(d)

    # Z^{(1 - s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(parity_12):
        qc.z(d)

    qc.reset(a)

    return qc

In [51]:
def add_HSH(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_ZY(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])

    # Y^{(1 - s0 s3)/2} odd parity
    parity_03 = expr.bit_xor(c[0], c[3])
    with qc.if_test(parity_03):
        qc.y(d)

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)

    qc.reset(a)

    return qc

In [52]:
def add_HS(qc, d, a, cbit):
    c = cbit

    measure_XI(qc, a, d, c[0])  # s0
    measure_ZY(qc, a, d, c[1])  # s1
    measure_ZZ(qc, a, d, c[2])  # s2
    measure_YI(qc, a, d, c[3])  # s3
    measure_XI(qc, a, d, c[4])  # s4

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)

    # Z^{(1 + s0 s1 s3)/2} even parity
    parity_013 = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[3])
    with qc.if_test(expr.logic_not(parity_013)):
        qc.z(d)

    qc.reset(a)

    return qc

## Single qubit gate in supremacy circuit

In [53]:
def add_sqrtX(qc, d, a, cbit):
    qc = add_HSH(qc, d, a, cbit)
    return qc

In [54]:
def add_sqrtY(qc, d, a, cbit):
   qc = add_S(qc, d, a, cbit)
   qc = add_HS(qc, d, a, cbit)
   return qc

In [55]:
def add_sqrtW(qc, d, a, cbit):
    qc.tdg(d)
    qc = add_sqrtX(qc, d, a, cbit)
    qc.t(d)
    return qc

## Two qubit gate in supremacy circuit

In [56]:
def add_CNOT(qc, c, a, t, cbit):
    # needs 8 cbits. all_ reuses cbit[0-4]. measure_ uses cbit[5-7]
    # Qubit A is initialized in an eigenstate of Z

    # prepare a in z-basis
    qc.reset(a)

    measure_ZX(qc, c, a, cbit[5])
    measure_ZX(qc, a, t, cbit[6])

    # this is single X-measurement on qubit a. h is on level of simulation, not circuit element
    qc.h(a)
    qc.measure(a, cbit[7])
    qc.h(a)

    # reset a to 0 for an easier comparision
    qc.reset(a)

    # we use cbit directly because it stores as 0(even) and 1(odd), as needed in paper
    P1 = cbit[5]
    P2 = cbit[6]
    M  = cbit[7]

    # X_t^{(P1 ⊕ M)}
    xt = expr.bit_xor(P1, M)
    with qc.if_test(xt):
        qc.x(t)

    #Z_c^{P2}
    with qc.if_test((P2, 1)):
        qc.z(c)

    return qc

In [57]:
# iswap(theta)^dagger decomposition

theta = Parameter("θ")

# H = 1/2 (XX + YY)
H = SparsePauliOp.from_list([
    ("XX", 0.5),
    ("YY", 0.5),
])

# This implements exp(-i * theta * H)
gate = PauliEvolutionGate(H, time=theta)
synth = SuzukiTrotter(order=1, reps=1)
decomposed = synth.synthesize(gate)

from qiskit import transpile

decomposed = transpile(
    decomposed,                # your circuit
    basis_gates=["cx", "rz", "h", "s", "sdg"],  # force decomposition
    optimization_level=0       # avoid simplification
)

print("iSWAP(theta)^dagger")
decomposed.draw()

iSWAP(theta)^dagger


┌───┐                   ┌───┐┌─────┐┌───┐                   ┌───┐┌───┐
q_0: ┤ H ├──■─────────────■──┤ H ├┤ Sdg ├┤ H ├──■─────────────■──┤ H ├┤ S ├
     ├───┤┌─┴─┐┌───────┐┌─┴─┐├───┤├─────┤├───┤┌─┴─┐┌───────┐┌─┴─┐├───┤├───┤
q_1: ┤ H ├┤ X ├┤ Rz(θ) ├┤ X ├┤ H ├┤ Sdg ├┤ H ├┤ X ├┤ Rz(θ) ├┤ X ├┤ H ├┤ S ├
     └───┘└───┘└───────┘└───┘└───┘└─────┘└───┘└───┘└───────┘└───┘└───┘└───┘

In [58]:
def add_iSWAPdg(qc, c, a, t, theta, cbit):

    # Here I use convention of supremacy paper
    # standard iswap has -θ/2, qiskit xx+yy has θ/2

    qc = add_H(qc, c, a, cbit)
    qc = add_H(qc, t, a, cbit)
    qc = add_CNOT(qc, c, a, t, cbit)
    qc.rz(theta,t)
    qc = add_CNOT(qc, c, a, t, cbit)
    qc = add_H(qc, c, a, cbit)
    qc = add_H(qc, t, a, cbit)

    # S^dagger ~ S^3
    qc = add_S(qc, c, a, cbit)
    qc = add_S(qc, c, a, cbit)
    qc = add_S(qc, c, a, cbit)
    qc = add_S(qc, t, a, cbit)
    qc = add_S(qc, t, a, cbit)
    qc = add_S(qc, t, a, cbit)

    qc = add_H(qc, c, a, cbit)
    qc = add_H(qc, t, a, cbit)
    qc = add_CNOT(qc, c, a, t, cbit)
    qc.rz(theta,t)
    qc = add_CNOT(qc, c, a, t, cbit)
    qc = add_SH(qc, c, a, cbit)
    qc = add_SH(qc, t, a, cbit)

    return qc

In [59]:
phi = Parameter("φ")

qc = QuantumCircuit(2)
qc.append(CPhaseGate(-phi), [0, 1])

decomposed = transpile(
    qc,
    basis_gates=["cx", "rz", "h", "s", "sdg", "sx"],
    optimization_level=0
)

qc.draw()
decomposed.draw()

global phase: (-0.25)*φ
     ┌──────────────┐                                       
q_0: ┤ Rz((-0.5)*φ) ├──■─────────────────■──────────────────
     └──────────────┘┌─┴─┐┌───────────┐┌─┴─┐┌──────────────┐
q_1: ────────────────┤ X ├┤ Rz(0.5*φ) ├┤ X ├┤ Rz((-0.5)*φ) ├
                     └───┘└───────────┘└───┘└──────────────┘

In [60]:
def add_CPhase(qc, c, a, t, phi, cbit):

    # Here I use convention of supremacy paper
    # standard cz has pi, qiskit cphase has -phi

    qc.rz(-0.5*phi, c)
    qc = add_CNOT(qc, c, a, t, cbit)
    qc.rz(0.5*phi, t)
    qc = add_CNOT(qc, c, a, t, cbit)
    qc.rz(-0.5*phi, t)

    return qc

In [61]:
def add_TwoQubitGate(qc, c, a, t, theta, phi, Z1, Z2, Z3, Z4, cbit):

    qc.rz(Z1, c)
    qc.rz(Z2, t)
    qc = add_iSWAPdg(qc, c, a, t, theta, cbit)
    qc = add_CPhase(qc, c, a, t, phi, cbit)
    qc.rz(Z3, c)
    qc.rz(Z4, t)

    return qc

In [62]:
def fSim(theta, phi):
    return np.array([
        [1, 0, 0, 0],
        [0, np.cos(theta), -1j*np.sin(theta), 0],
        [0, -1j*np.sin(theta), np.cos(theta), 0],
        [0, 0, 0, np.exp(-1j*phi)]
    ], dtype=complex)

# Test MBQC gates

For each gate, we test on 100 individual trials

For each trial:

- Prepare a **random state over the full Hilbert space**
- Use **random parameters** when applicable
- Execute both MBQC and GBQC circuits
- Compare the final states and check the overlap
$
\left| \langle \psi_{\mathrm{GBQC}} \mid \psi_{\mathrm{MBQC}} \rangle \right| = 1
$

---

This method ensures:

1. **Eliminates relative phase ambiguity**  

2. **Covers the full Hilbert space**  

3. **Samples measurement branches**  

4. **Validates arbitrary parameters**  

## Two-qubit gate test

In [63]:
sim = AerSimulator(method="statevector")

num_trials = 100
tol = 1e-6

all_pass = True

for i in range(num_trials):

    psi02 = random_statevector(4)
    phi = random.uniform(0, 2*pi)
    theta = random.uniform(0, 2*pi)

    # -----------------------
    # DIRECT (your matrix)
    # -----------------------
    qc1 = QuantumCircuit(3)
    qc1.initialize(psi02.data, [0, 2])

    U2 = UnitaryGate(fSim(theta, phi))
    qc1.append(U2, [0, 2])

    sv1 = Statevector.from_instruction(qc1)

    # -----------------------
    # MBQC (your circuit)
    # -----------------------
    qc2 = QuantumCircuit(3)
    cbit2 = ClassicalRegister(8)
    qc2.add_register(cbit2)

    qc2.initialize(psi02.data, [0, 2])

    qc2 = add_TwoQubitGate(
        qc2,
        0,  # c
        1,  # a (ancilla)
        2,  # t
        theta,
        phi,
        0, 0, 0, 0,
        cbit2
    )

    qc2.save_statevector(conditional=True)

    result = sim.run(qc2, shots=1).result()
    data = result.data(0)["statevector"]

    sv_mbqc = list(data.values())[0]

    # -----------------------
    # Compare
    # -----------------------
    overlap = abs(np.vdot(sv1.data, sv_mbqc.data))

    if abs(overlap - 1) > tol:
        print(f"❌ FAIL at trial {i+1}: overlap = {overlap}")
        all_pass = False
        break

if all_pass:
    print("✅ All fSim tests pass (overlap ≈ 1)")

✅ All fSim tests pass (overlap ≈ 1)


## Single-qubit gate test

In [64]:
sim = AerSimulator(method="statevector")

num_trials = 100
tol = 1e-6

all_pass = True

for i in range(num_trials):

    # --- random input state (1 qubit on data wire) ---
    psi = random_statevector(2)

    # --- gate-based (GBQC) ---
    qc1 = QuantumCircuit(2)
    qc1.initialize(psi.data, 0)
    qc1.sx(0)

    sv1 = Statevector.from_instruction(qc1)

    # --- MBQC ---
    qc2 = QuantumCircuit(2)
    cbit2 = ClassicalRegister(5)
    qc2.add_register(cbit2)

    qc2.initialize(psi.data, 0)
    qc2 = add_sqrtX(qc2, 0, 1, cbit2)

    qc2.save_statevector(conditional=True)

    result = sim.run(qc2, shots=1).result()
    data = result.data(0)["statevector"]

    # extract one branch
    sv_mbqc = list(data.values())[0]

    # compute overlap
    overlap = abs(np.vdot(sv1.data, sv_mbqc.data))

    # check
    if abs(overlap - 1) > tol:
        print(f"❌ FAIL at trial {i+1}: overlap = {overlap}")
        all_pass = False
        break

if all_pass:
    print("✅ All randomized sqrt(X) tests passed (overlap ≈ 1)")

✅ All randomized sqrt(X) tests passed (overlap ≈ 1)


In [65]:
sim = AerSimulator(method="statevector")

num_trials = 100
tol = 1e-6

all_pass = True

for i in range(num_trials):

    # --- random input state (1 qubit on data wire) ---
    psi = random_statevector(2)

    # --- gate-based (GBQC) ---
    qc1 = QuantumCircuit(2)
    qc1.initialize(psi.data, 0)
    qc1.ry(np.pi/2, 0)

    sv1 = Statevector.from_instruction(qc1)

    # --- MBQC ---
    qc2 = QuantumCircuit(2)
    cbit2 = ClassicalRegister(5)
    qc2.add_register(cbit2)

    qc2.initialize(psi.data, 0)
    qc2 = add_sqrtY(qc2, 0, 1, cbit2)

    qc2.save_statevector(conditional=True)

    result = sim.run(qc2, shots=1).result()
    data = result.data(0)["statevector"]

    # extract one branch
    sv_mbqc = list(data.values())[0]

    # compute overlap
    overlap = abs(np.vdot(sv1.data, sv_mbqc.data))

    # check
    if abs(overlap - 1) > tol:
        print(f"❌ FAIL at trial {i+1}: overlap = {overlap}")
        all_pass = False
        break

if all_pass:
    print("✅ All randomized sqrt(Y) tests passed (overlap ≈ 1)")

✅ All randomized sqrt(Y) tests passed (overlap ≈ 1)


# Random circuit with single qubit gate

qubit geometry: $(i,0)=\text{data},\ (i,1)=\text{ancilla},\ (i,j)=2i+j$

Test with 4*2 qubits and 100 periods: random entangled state, random seed

In [66]:
def random_single_MBQC(qc, N, periods, seed):
    random.seed(seed)

    #all circuits with single-qubit clifford gates need at most 5 classical bits
    cbit = ClassicalRegister(8)

    qc.add_register(cbit)

    last_gate = [None] * N

    for p in range(periods):
        for i in range(N):

            choices = ["sX", "sY", "sW"]
            if last_gate[i] is not None:
                choices.remove(last_gate[i])
            gate = random.choice(choices)

            # apply
            if gate == "sX":
                qc = add_sqrtX(qc, 2*i, 2*i+1, cbit)
            elif gate == "sY":
                qc = add_sqrtY(qc, 2*i, 2*i+1, cbit)
            else:
                qc = add_sqrtW(qc, 2*i, 2*i+1, cbit)

            last_gate[i] = gate

    return qc

In [67]:
def random_single_gateQC(qc, N, periods, seed):
    random.seed(seed)

    last_gate = [None] * N

    for p in range(periods):
        for i in range(N):

            choices = ["sX", "sY", "sW"]
            if last_gate[i] is not None:
                choices.remove(last_gate[i])
            gate = random.choice(choices)

            # apply
            if gate == "sX":
                qc.sx(2*i)
            elif gate == "sY":
                qc.ry(np.pi/2, 2*i)
            else:
                qc.tdg(2*i)
                qc.sx(2*i)
                qc.t(2*i)

            last_gate[i] = gate

    return qc

In [69]:
sim = AerSimulator(method="statevector")

N = 4
periods = 100
num_trials = 10

print(f"single-qubit-gate circuit with 4*2 qubits and 100 periods")

for i in range(num_trials):

    psi = random_statevector(2**N)

    seed = np.random.randint(1000)

    # --- direct circuit ---
    qc1 = QuantumCircuit(2*N)
    qc1.initialize(psi.data, list(range(0, 2*N, 2)))
    qc1 = random_single_gateQC(qc1, N, periods, seed)
    sv1 = Statevector.from_instruction(qc1)

    # --- MBQC circuit ---
    qc2 = QuantumCircuit(2*N)
    qc2.initialize(psi.data, list(range(0, 2*N, 2)))
    qc2 = random_single_MBQC(qc2, N, periods, seed)

    qc2.save_statevector(conditional=True)

    result = sim.run(qc2, shots=1).result()
    data = result.data(0)["statevector"]

    # extract branch
    sv_mbqc = list(data.values())[0]

    # compute overlap
    overlap = abs(np.vdot(sv1.data, sv_mbqc.data))


    print(f"Trial {i+1}: overlap = {overlap}")

single-qubit-gate circuit with 4*2 qubits and 100 periods
Trial 1: overlap = 0.999999999999987
Trial 2: overlap = 0.9999999999999858
Trial 3: overlap = 0.9999999999999871
Trial 4: overlap = 0.9999999999999865
Trial 5: overlap = 0.9999999999999866
Trial 6: overlap = 0.9999999999999869
Trial 7: overlap = 0.9999999999999861
Trial 8: overlap = 0.999999999999986
Trial 9: overlap = 0.9999999999999869
Trial 10: overlap = 0.9999999999999855
